In [5]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

###### **idk why is it giving us the future warning but anyways , lets continue the process**

###### **This project is about predicting the laptop prices with the help of the features provided in the dataset , and the dataset is not preprocessed so we have to do it in the process**

In [6]:
  df = pd.read_csv("/kaggle/input/datasets/hemachander002/laptopdata-csv/laptopData.csv")
  df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/hemachander002/laptopdata-csv/laptopData.csv'

In [ ]:
df = df.drop(columns ="Unnamed: 0")
df.reset_index(inplace = True)

In [ ]:
df.drop(columns = "index",inplace = True)

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

###### **well... we have 30 null data in every columns and its not even 5 percent of the whole dataset so Im gonna drop the rows that has null values**

In [ ]:
df = df.dropna()
df.isnull().sum()

In [ ]:
for col in df.columns:
    if (col != "Price") and (col != "Weight"):
        print(col,df[col].unique())
        print()
    else:
        continue

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.boxplot(x = 'Price' , data = df)
plt.show()


##### **It may look like it has outliers but practically, laptops with higher configuration can be expensive as hell like alienware, acer predators**

In [ ]:
df.select_dtypes(include='object').head()

In [ ]:
df.columns = df.columns.str.strip()

for col in df.select_dtypes(include='object'):
    df[col] = df[col].str.strip()

In [ ]:
df["Weight"].unique()[:20]

In [ ]:
def clean_weight(x):
    try:
        return float(str(x).lower().replace('kg', '').strip())
    except:
        return None

df["Weight"] = df["Weight"].apply(clean_weight)

In [ ]:
df.head()

In [ ]:
df["Price"] = np.round(df["Price"],2)

In [ ]:
sns.displot(x = "Price" , data = df)
plt.title("Price Distribution")
plt.show()

In [ ]:
df["Price"].mean()

In [ ]:
def clean_ram(x):
    try:
        return float(str(x).lower().replace('gb', '').strip())
    except:
        return None

df["Ram"] = df["Ram"].apply(clean_ram)

In [ ]:
print(f"before ? removal : {df.info} ")
print()
df = df.replace('?', np.nan).dropna()
print(f"after ? removal : {df.info} ")



In [ ]:
# lol i cud have just used df.shape() . seems like only 3 rows had "?" lets crosscheck again to satisfy my overthinking 
for col in df.columns:
    if (col != "Price") and (col != "Weight"):
        print(col,df[col].unique())
        print()
    else:
        continue

In [ ]:
df["Inches"] = df["Inches"].astype(float)
df.dtypes

In [ ]:
#lets look at our dataframe again
df.tail()

In [ ]:
#lets crosscheck again before we begin feature engineering
df.info()

In [ ]:
df.isin(['?']).any(axis=1).sum()
df["Ram"] = df["Ram"].astype(int)

In [ ]:
df["touch"] = df["ScreenResolution"].str.contains(r'touch\s*screen', case = False , na = False).astype(int)

In [ ]:
df[df['ScreenResolution'].str.contains(r'touch\s*screen', case=False, na=False)]["touch"].unique()

In [ ]:
# okay .. now it is pakka lets look at our dataframe again
df[df['ScreenResolution'].str.contains(r'touch\s*screen', case=False, na=False)]

In [ ]:
# lets check the df's statistical description to get some basic insights
df.describe()

In [ ]:
# im feeling sussy on the weights and price column ,
# coz i have never seen a laptop that weighs only 2 grams and 11 kg , who the hell in the world carries a 11kg thing on their back 

df[((df["Weight"] < 0.9)|(df["Weight"] > 3.6))]

In [ ]:
df = df[(df["Weight"] > 0.9) & (df["Weight"] < 3.6)]
df.describe()

In [ ]:
# now lets do the same with the price column too ,
# based on my research lowest price of a laptop can be in between  
# 9k to 14k , there are some intel celron laptops and chromebooks that are way cheaper

df[df["Price"] < 10000]

In [ ]:
# well it makes sense , so lets dont touch the price column again

In [ ]:
#lets extract only the resolution part
df['resolution'] = df['ScreenResolution'].str.extract(r'(\d+\s*x\s*\d+)')
df.head()

In [ ]:
# lets check if it had worked correctly
df.info()

In [ ]:
df["resolution"].unique()

In [ ]:
#now , we dont want the screen resolution column so ,lets ditch that column
df.drop("ScreenResolution",axis = 1 , inplace = True)
df.head()

In [ ]:
# maybe we dont need the inches part too coz we can guess the size of the laptop just with the resolution so lets ditch this column too
df.drop("Inches" , axis = 1 , inplace = True)
df.head()

In [ ]:
df["amd"] = df["Cpu"].str.contains(r'\bamd\b' , case = False , na = False).astype(int)
df.info()

In [ ]:
df[df["Cpu"].str.contains(r'\bamd\b' , case = False , na = False)]['amd'].unique()

In [ ]:
# lets do the same for intel
df["intel"] = df["Cpu"].str.contains(r'\bintel\b' , case = False , na = False).astype(int)
df[df["Cpu"].str.contains(r'\bintel\b' , case = False , na = False)]['intel'].unique()


In [ ]:
df.head()

In [ ]:
def cpu_category(cpu):
    cpu = str(cpu).lower()
    
    # based on my research these are the keywords for the high end cpus
    if any(x in cpu for x in ['i7', 'i9', 'xeon', 'hq', 'hk', 'ryzen 7', 'ryzen 9', 'fx']):
        return 'high'
    
    # these are the mid ones 
    elif any(x in cpu for x in ['i5', 'i3', 'ryzen 3', 'ryzen 5', 'a8', 'a10', 'a12']):
        return 'mid'
    
    # rest of them are going to be considered as budget 
    else:
        return 'budget'

df['cpu_range'] = df['Cpu'].apply(cpu_category)

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df[df["Cpu"].str.contains(r'\bamd\b' , case = False , na = False)][df["cpu_range"] == "mid"]

In [ ]:
# we have got enough info from the cpu column and i dont think we need that column again for future use so , lets ditch that 
df.drop("Cpu",inplace = True , axis = 1)
df.head()

In [ ]:
df = pd.get_dummies(df, columns=["cpu_range"])

In [ ]:
df.head()

In [ ]:
#now lets work on the Memory column , im gonna use chatgpt to help me sort this out-
def convert_storage(x):
    import re
    total = 0
    
    matches = re.findall(r'(\d+\.?\d*)(TB|GB)', str(x))
    
    for num, unit in matches:
        num = float(num)
        if unit == 'TB':
            num *= 1024   # convert TB → GB
        total += num
        
    return total

df['storage'] = df['Memory'].apply(convert_storage)
df.head()


In [ ]:
df["ssd"] = df["Memory"].str.contains(r'\bSSD\b',case = False , na = False).astype(int)
df["hdd"] = df["Memory"].str.contains(r'\bHDD\b', case=False, na=False).astype(int)
df["hybrid"] = df["Memory"].str.contains(r'\bHybrid\b', case=False, na=False).astype(int)
df["flash"] = df["Memory"].str.contains(r'\bFlash\b', case=False, na=False).astype(int)
df["hdd+ssd"] = (
    df["Memory"].str.contains(r'\bSSD\b', case=False, na=False) &
    df["Memory"].str.contains(r'\bHDD\b', case=False, na=False)
).astype(int)


In [ ]:
df[(
    df["Memory"].str.contains(r'\bSSD\b', case=False, na=False) &
    df["Memory"].str.contains(r'\bHDD\b', case=False, na=False)
)]["hdd+ssd"].unique()

In [ ]:
#i think we have extracted most of the info from the storage section so lets ditch the memory column
df.drop("Memory",axis = 1 , inplace = True)
df.head()

In [ ]:
df["storage"] = df["storage"].astype(int)
df.head()

In [ ]:
df = pd.get_dummies(df, columns=['OpSys'])
df.head()

In [ ]:
# I planned to one hot encode the resolution column but later i decided to create 2 columns from the resolution 
# and calculate the pixel count so that our dataset will be having less dimensions
df[['width', 'height']] = df['resolution'].str.split('x', expand=True).astype(int)
df["pixels"] = (df["height"] * df["width"])
df.drop("resolution",axis = 1 , inplace = True)
df.head()

In [ ]:
df.columns

In [ ]:
df = df.rename(columns={
    'OpSys_Android': "android",
    'OpSys_Chrome OS' : "chrome",
    'OpSys_Linux' : "linux", 
    'OpSys_Mac OS X' : "macX", 
    'OpSys_No OS' : "no_os",
    'OpSys_Windows 10' : "win10", 
    'OpSys_Windows 10 S' : "win10_s", 
    'OpSys_Windows 7' : "win7",
    'OpSys_macOS' : "mac_os"
})
df.info()

In [ ]:
# xD i shud have done this earlier
df.columns = df.columns.str.lower()
df.head()

In [ ]:
# now lets work on the gpu part
import re

def gpu_rank(gpu):
    gpu = str(gpu).lower()
    
    # INTEL  == always the low budget ones
    if 'intel' in gpu:
        return 5
    
    # NVIDIA
    if 'nvidia' in gpu or 'geforce' in gpu:
        
        # RTX cards ... well 40 
        #and 50 series cards are expensive and we dont have 
        #the enough data to predict the 40 and 50 series laptop's price so im ignoring those cards
        if 'rtx' in gpu:
            num = re.findall(r'\d{2,3}', gpu)
            if num:
                last_two = int(num[0]) % 100
                
                if last_two >= 80: return 1
                elif last_two >= 70: return 2
                elif last_two >= 60: return 3
                elif last_two >= 50: return 4
            return 2  # default RTX
        
        # GTX cards
        if 'gtx' in gpu:
            num = re.findall(r'\d{1,9}', gpu)
            if num:
                last_two = int(num[0]) % 100
                
                if last_two >= 80: return 1
                elif last_two >= 70: return 2
                elif last_two >= 60: return 3
                elif last_two >= 50: return 4
                elif last_two >= 40: return 5
                elif last_two >= 30: return 5
        
        # MX and GT are the low ones .. coz i was a gt user back then :( i know the pain
        if 'mx' in gpu or 'gt' in gpu:
            return 5
        
        if 'quadro' in gpu:
            return 1
        
        return 5
    
    # AMD i am unsure of AMD cards coz i ve nver been an AMD consumer so based on our data. so i admit that my model works the best with the Nvidia cards
    if 'amd' in gpu or 'radeon' in gpu:
        
        if 'rx' in gpu:
            num = re.findall(r'\d{5}', gpu)
            if num:
                last_two = int(num[0]) % 100
                
                if last_two >= 80: return 1
                elif last_two >= 90: return 1
                elif last_two >= 70: return 2
                elif last_two >= 60: return 3
                elif last_two >= 50: return 4
        
        # Old R series
        if 'r7' in gpu: return 3
        if 'r5' in gpu: return 4
        if 'r3' in gpu: return 5
        if 'r2' in gpu: return 5
        if 'r4' in gpu: return 5

        if 'firepro' in gpu: return 1
        
        return 4
    
    return 5

In [ ]:
df["gpu_rank"] = df["gpu"].apply(gpu_rank)
df.head()

In [ ]:
df.rename(columns={"amd": "amd_cpu", "intel": "intel_cpu"}, inplace=True)

In [ ]:
df.head()

In [ ]:
# we have extracted the maximum info from the gpu column so lets delete that column from our dataset
df.drop("gpu",inplace = True,axis = 1)
df.info()

In [ ]:
df.head()


In [ ]:
# to avoid multicollinearity , lets remove one column from the cpu sector and one column from the budget 
df.drop(["amd_cpu","cpu_range_budget"],axis = 1,inplace= True)


In [ ]:
# and i forgot to remove the h and w columns coz we have alrdy derived the pixels column from the so lets remove both of them
df.drop(["height","width"],axis = 1,inplace= True)
df.head()

In [ ]:
df.info()

In [ ]:
df = pd.get_dummies(df,columns=["company","typename"])
df.head()

In [ ]:
df.columns

In [ ]:
df.rename(columns = {'company_Acer' :'acer', 
                'company_Apple' : 'apple',
       'company_Asus' : "asus", 
       'company_Chuwi' : "chuwi",
        'company_Dell' : "dell", 
        'company_Fujitsu' : "fujitsu",
       'company_Google' : "google", 
       'company_HP' : "hp",
         'company_Huawei' : "huawei", 
         'company_LG' : "lg",
       'company_Lenovo' : "lenovo", 
       'company_MSI' : "msi", 
       'company_Mediacom' : "mediacom",
       'company_Microsoft' : "microsoft", 
       'company_Razer' : "razer",
        'company_Samsung' : "samsung",
       'company_Toshiba' : "toshiba",
        'company_Vero' : "vero",
         'company_Xiaomi' : "xiaomi",
       'typename_2 in 1 Convertible' : "2_in_1",
        'typename_Gaming' : "gaming",
         'typename_Netbook' : "netbook",
       'typename_Notebook' : "notebook",
        'typename_Ultrabook' : "ultrabook", 
        'typename_Workstation' : "workstation"
    
},inplace = True)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.rename(columns = {"hdd+ssd" : "hdd_ssd"} , inplace = True)

In [ ]:
#alright we have reached the end of the preprocessing stage ..
#now all we have to do is remove some columns from the one hot encoded columns so that we can avoid the redundancy issues

df.drop(['no_os', 'samsung', 'netbook'], axis=1, inplace=True)
df.head()


In [ ]:
#lets have a glance at the df description , before we start modelling it
df.describe()

In [ ]:
df.info()

In [ ]:
# now we are all set for modelling , lets jump into scikit-learn and shi

X = df.drop('price', axis=1)
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# we need to normalize the values before applying linear models

num_cols = ['ram', 'weight', 'storage', 'pixels',"gpu_rank"]
bin_cols = [col for col in X.columns if col not in num_cols]

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_num = scaler.fit_transform(X_train[num_cols])
X_test_num = scaler.transform(X_test[num_cols])

X_train_final = np.hstack([X_train_num, X_train[bin_cols].values])
X_test_final = np.hstack([X_test_num, X_test[bin_cols].values])

In [ ]:
#linear regression

lr = LinearRegression()
lr.fit(X_train_final, y_train)
y_pred_lr = lr.predict(X_test_final)

In [ ]:
# Linear regression with L2 regularization 
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_final, y_train)
y_pred_ridge = ridge.predict(X_test_final)

In [ ]:
# Linear regression with L1 regularization
from sklearn.linear_model import LassoCV

lasso = LassoCV(cv=5, max_iter=50000)
lasso.fit(X_train_final, y_train)
y_pred_lasso = lasso.predict(X_test_final)

In [ ]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05
)

xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

In [ ]:
# as we are dealing with the regression models , r2 score and the mae 
from sklearn.metrics import r2_score, mean_absolute_error,mean_squared_error

def evaluate(y_test, y_pred, name):
    print(f"{name}")
    print("R2:", r2_score(y_test, y_pred))
    print("MAE:", mean_absolute_error(y_test, y_pred))
    print("MSE:", mean_squared_error(y_test, y_pred))
    print("------------")

evaluate(y_test, y_pred_lr, "Linear")
evaluate(y_test, y_pred_ridge, "Ridge")
evaluate(y_test, y_pred_lasso, "Lasso")
evaluate(y_test, y_pred_xgb, "XGBoost")

In [ ]:
# as we can see and as we expected , xgbregressor worked the best among the others , but linear models performed suprisingly better
# if i tune my xgb model , it wud get better r2 scores . so lets tune our model

xgb = XGBRegressor(
    n_estimators=800,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.1,
    reg_lambda=1,
    random_state=42
)

xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

In [ ]:
# lets run the previous block of code again 
from sklearn.metrics import r2_score, mean_absolute_error,mean_squared_error

def evaluate(y_test, y_pred, name):
    print(f"{name}")
    print("R2:", r2_score(y_test, y_pred))
    print("MAE:", mean_absolute_error(y_test, y_pred))
    print("MSE:", mean_squared_error(y_test, y_pred))
    print("------------")

evaluate(y_test, y_pred_lr, "Linear")
evaluate(y_test, y_pred_ridge, "Ridge")
evaluate(y_test, y_pred_lasso, "Lasso")
evaluate(y_test, y_pred_xgb, "XGBoost")

In [ ]:
# hmm , we can see some improvement in the r2 score and minute improvement in the MAE . still im not satisfied with the results so lets tune again
xgb = XGBRegressor(
    n_estimators=1200,
    max_depth=7,
    learning_rate=0.02,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    random_state=42
)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

In [ ]:
# lets run the previous block of code again 
from sklearn.metrics import r2_score, mean_absolute_error,mean_squared_error

def evaluate(y_test, y_pred, name):
    print(f"{name}")
    print("R2:", r2_score(y_test, y_pred))
    print("MAE:", mean_absolute_error(y_test, y_pred))
    print("MSE:", mean_squared_error(y_test, y_pred))
    print("------------")

evaluate(y_test, y_pred_lr, "Linear")
evaluate(y_test, y_pred_ridge, "Ridge")
evaluate(y_test, y_pred_lasso, "Lasso")
evaluate(y_test, y_pred_xgb, "XGBoost")

In [ ]:
# wow , now out xgboost model working better than the rest , but still , i wanna tune it even more just to satisfy my curiousity

from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRegressor

param_dist = {
    'n_estimators': [500, 800, 1200],
    'max_depth': [5, 6, 7, 8, 10],
    'learning_rate': [0.01, 0.02, 0.05],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'gamma': [0, 0.1, 0.2],
    'min_child_weight': [1, 3, 5]
}

xgb = XGBRegressor(random_state=42)

random_search = RandomizedSearchCV(
    xgb,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_train, y_train)

print(random_search.best_params_)

In [ ]:
# we got the best parameters from the randomized search cv
xgb = XGBRegressor(
    n_estimators=1200,
    max_depth=10,
    learning_rate=0.01,
    subsample=0.7,
    colsample_bytree=0.8,
    min_child_weight=1,
    gamma=0.2,
    random_state=42
)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error,mean_squared_error

def evaluate(y_test, y_pred, name):
    print(f"{name}")
    print("R2:", r2_score(y_test, y_pred))
    print("MAE:", mean_absolute_error(y_test, y_pred))
    print("MSE:", mean_squared_error(y_test, y_pred))
    print("------------")

evaluate(y_test, y_pred_lr, "Linear")
evaluate(y_test, y_pred_ridge, "Ridge")
evaluate(y_test, y_pred_lasso, "Lasso")
evaluate(y_test, y_pred_xgb, "XGBoost")

In [ ]:
#lets try again after removing the so called outliers in the price column 
df2  = df[df["price"] < df["price"].quantile(0.99)]
df2.describe()

In [ ]:
X2 = df2.drop('price', axis=1)
y2 = df2['price']

x_train, x_test, y_train2, y_test2 = train_test_split(
    X2, y2, test_size=0.2, random_state=42
)

In [ ]:
xgb = XGBRegressor(
    n_estimators=1200,
    max_depth=10,
    learning_rate=0.01,
    subsample=0.7,
    colsample_bytree=0.8,
    min_child_weight=1,
    gamma=0.2,
    random_state=42
)
xgb.fit(x_train, y_train2)
y_pred_xgb2 = xgb.predict(x_test)

In [ ]:
scaler2 = StandardScaler()

num_cols2 = ['ram', 'weight', 'storage', 'pixels',"gpu_rank"]
bin_cols2 = [col for col in X.columns if col not in num_cols]

x_train_num2 = scaler2.fit_transform(x_train[num_cols])
x_test_num2 = scaler2.transform(x_test[num_cols])

x_train_final = np.hstack([x_train_num2, x_train[bin_cols].values])
x_test_final = np.hstack([x_test_num2, x_test[bin_cols].values])

lr = LinearRegression()
lr.fit(x_train_final, y_train2)
y_pred_lr2 = lr.predict(x_test_final)

ridge = Ridge(alpha=1.0)
ridge.fit(x_train_final, y_train2)
y_pred_ridge2 = ridge.predict(x_test_final)

lasso = LassoCV(cv=5, max_iter=50000)
lasso.fit(x_train_final, y_train2)
y_pred_lasso2 = lasso.predict(x_test_final)


In [ ]:
def evaluate(y_test, y_pred, name):
    print(f"{name}")
    print("R2:", r2_score(y_test, y_pred))
    print("MAE:", mean_absolute_error(y_test, y_pred))
    print("MSE:", mean_squared_error(y_test, y_pred))
    print("------------")

evaluate(y_test2, y_pred_lr2, "Linear")
evaluate(y_test2, y_pred_ridge2, "Ridge")
evaluate(y_test2, y_pred_lasso2, "Lasso")
evaluate(y_test2, y_pred_xgb2, "XGBoost")